In [ ]:
/**
 https://www.bambooweekly.com/egg-prices/
 https://www.bambooweekly.com/egg-prices-solution/
 https://github.com/JoergEm/Bamboo-Weekly/tree/main 
*/

In [ ]:
// Imports
%useLatestDescriptors
%use dataframe
@file:DependsOn("org.jetbrains.kotlinx:dataframe-excel:0.8.0")
@file:DependsOn("org.apache.logging.log4j:log4j-api:2.24.3")
@file:DependsOn("org.apache.logging.log4j:log4j-core:2.24.3")
import org.jetbrains.kotlinx.dataframe.api.*
import java.io.File
import java.io.InputStream
import java.io.OutputStream
import java.net.URL
import java.nio.file.Path
import java.nio.file.Paths
println("Imports ✅")

In [ ]:
// Function creating local folders
fun create_folders(folderNames: List<String>, basePath: String = ".") {
    folderNames.forEach { folderName ->
        val dir = File(basePath, folderName)
        if (dir.mkdirs()) {
            println("Folders ✅")
        } else if (dir.exists()) {
            println("Folders ✅")
        } else {
            println("Error ❌")
            ""
        }
    }
}

In [ ]:
// Function creating paths for downloads
fun joinPath(vararg paths: String): String {
    return try {
        var path: Path = Paths.get(paths[0])
        for (i in 1 until paths.size) {
            path = path.resolve(paths[i])
        }
        path.toString()
    } catch (e: Exception) {
        println("Error ❌")
        ""
    }
}

In [ ]:
// Function downloading data locally
fun download_data(url: String, outputFile: String) {
    try {
        val inputStream: InputStream = URL(url).openStream()
        val outputStream: OutputStream = File(outputFile).outputStream()
        inputStream.use { input ->
            outputStream.use { output ->
                input.copyTo(output)
            }
        }
        println("Data ✅")
    } catch (e: Exception) {
        println("Error ❌")
        ""
    }
}

In [ ]:
val data: DataFrame

if (!Files.exists(filepath)) {
    try {
        // Annahme: getData(years) existiert bereits
        data = getData(years)

        // Excel schreiben
        val workbook = XSSFWorkbook()
        val sheet = workbook.createSheet("data")

        // Header
        val headerRow = sheet.createRow(0)
        data.names().forEachIndexed { i, name ->
            headerRow.createCell(i).setCellValue(name)
        }

        // Rows
        data.rows().forEachIndexed { r, row ->
            val excelRow = sheet.createRow(r + 1)
            data.names().forEachIndexed { c, col ->
                excelRow.createCell(c).setCellValue(row[col]?.toString())
            }
        }

        Files.newOutputStream(filepath).use {
            workbook.write(it)
        }
        workbook.close()

        display(Markdown("Data ✅"))
    } catch (e: Exception) {
        display(Markdown("Error ❌"))
        throw e
    }
} else {
    // Excel lesen
    val workbook = XSSFWorkbook(Files.newInputStream(filepath))
    val sheet = workbook.getSheetAt(0)

    val headers = sheet.getRow(0).map { it.stringCellValue }

    val rows = sheet.drop(1).map { row ->
        headers.mapIndexed { i, col ->
            col to row.getCell(i)?.toString()
        }.toMap()
    }

    workbook.close()

    data = DataFrame.fromRows(rows)
    display(Markdown("Data loaded from existing file ✅"))
}

In [ ]:
// Given URLs for each of the last five years of egg prices, create a single data frame.
df.take(5)

In [ ]:
// What was the average low price for eggs in each of the years o…
data
    .convert("Date").toLocalDate()
    .add("Year") { it["Date"].map { d -> (d as LocalDate).year } }
    .groupBy("Year")
    .mean()
